In [3]:
import os
import pandas as pd

def read_json_files_to_dataframe(directory):
    dataframes = []
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            try:
                # Read JSON file into DataFrame
                df = pd.read_json(file_path)
                df['source_file'] = filename  # Add column to track source
                dataframes.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    # Combine all DataFrames into one
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    else:
        print("No JSON files found or failed to read.")
        return pd.DataFrame()

# Example usage
directory_path = './out'  # Update to your path
result_df = read_json_files_to_dataframe(directory_path)
result_df.head()

,SpotPriceHistory,source_file,BillingCurrency,CustomerEntityId,CustomerEntityType,Items,NextPageLink,Count
0,"{'AvailabilityZone': 'us-east-1c', 'InstanceTy...",aws.json,NaN,NaN,NaN,NaN,NaN,NaN
1,"{'AvailabilityZone': 'us-east-1b', 'InstanceTy...",aws.json,NaN,NaN,NaN,NaN,NaN,NaN
2,"{'AvailabilityZone': 'us-east-1d', 'InstanceTy...",aws.json,NaN,NaN,NaN,NaN,NaN,NaN
3,"{'AvailabilityZone': 'us-east-1f', 'InstanceTy...",aws.json,NaN,NaN,NaN,NaN,NaN,NaN
4,"{'AvailabilityZone': 'us-east-1a', 'InstanceTy...",aws.json,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import os
import pandas as pd
import json

def read_json_files_to_dataframe(directory):
    dataframes = []
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as file:
                    data = json.load(file)
                    
                    # Handle specific JSON structures
                    if 'Items' in data:
                        df = pd.DataFrame(data['Items'])
                        df['BillingCurrency'] = data.get('BillingCurrency', None)
                        df['CustomerEntityId'] = data.get('CustomerEntityId', None)
                        df['CustomerEntityType'] = data.get('CustomerEntityType', None)
                    elif 'SpotPriceHistory' in data:
                        df = pd.DataFrame(data['SpotPriceHistory'])
                    else:
                        df = pd.DataFrame([data])
                    
                    df['source_file'] = filename  # Add column to track source
                    dataframes.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    # Combine all DataFrames into one
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    else:
        print("No JSON files found or failed to read.")
        return pd.DataFrame()

def find_lowest_spot_price(df):
    spot_price_df = df[df['source_file'] == 'aws.json']
    if 'SpotPrice' in spot_price_df.columns:
        spot_price_df['SpotPrice'] = pd.to_numeric(spot_price_df['SpotPrice'])
        lowest_price_row = spot_price_df.loc[spot_price_df['SpotPrice'].idxmin()]
        print("Lowest Spot Price:")
        print(lowest_price_row)
    else:
        print("No Spot Price data found.")

# Example usage
directory_path = './out'  # Update to your path
result_df = read_json_files_to_dataframe(directory_path)
# print(result_df.head())
find_lowest_spot_price(result_df)


Lowest Spot Price:
AvailabilityZone                       us-east-1a
InstanceType                             m5.large
ProductDescription                     Linux/UNIX
SpotPrice                                  0.0291
Timestamp               2025-03-20T01:32:34+00:00
source_file                              aws.json
currencyCode                                  NaN
tierMinimumUnits                              NaN
retailPrice                                   NaN
unitPrice                                     NaN
armRegionName                                 NaN
location                                      NaN
effectiveStartDate                            NaN
effectiveEndDate                              NaN
meterId                                       NaN
meterName                                     NaN
productId                                     NaN
skuId                                         NaN
productName                                   NaN
skuName                        

/tmp/ipykernel_6984/2704530977.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spot_price_df['SpotPrice'] = pd.to_numeric(spot_price_df['SpotPrice'])


In [8]:
import os
import pandas as pd
import json

def read_json_files_to_dataframe(directory):
    dataframes = []
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as file:
                    data = json.load(file)
                    
                    # Handle specific JSON structures
                    if 'Items' in data:
                        df = pd.DataFrame(data['Items'])
                        df['BillingCurrency'] = data.get('BillingCurrency', None)
                        df['CustomerEntityId'] = data.get('CustomerEntityId', None)
                        df['CustomerEntityType'] = data.get('CustomerEntityType', None)

                        # Convert Azure unit prices to per-second prices if unit is in hours
                        df['NormalizedUnitPrice'] = df.apply(
                            lambda x: x['unitPrice'] / 3600 if x.get('unitOfMeasure') == '1 Hour' else x['unitPrice'], axis=1
                        )
                    elif 'SpotPriceHistory' in data:
                        df = pd.DataFrame(data['SpotPriceHistory'])
                        df['SpotPrice'] = pd.to_numeric(df['SpotPrice'])
                    else:
                        df = pd.DataFrame([data])
                    
                    df['source_file'] = filename  # Add column to track source
                    dataframes.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    # Combine all DataFrames into one
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    else:
        print("No JSON files found or failed to read.")
        return pd.DataFrame()

def find_lowest_spot_price(df):
    # Extract AWS and Azure data
    aws_df = df[df['source_file'] == 'aws.json']
    azure_df = df[df['source_file'] == 'azure.json']

    # Find lowest SpotPrice for AWS
    lowest_aws_price = aws_df.loc[aws_df['SpotPrice'].idxmin()] if not aws_df.empty else None

    # Find lowest NormalizedUnitPrice for Azure
    lowest_azure_price = azure_df.loc[azure_df['NormalizedUnitPrice'].idxmin()] if not azure_df.empty else None

    print("Lowest Spot Price for AWS:")
    print(lowest_aws_price if lowest_aws_price is not None else "No AWS data available.")

    print("\nLowest Spot Price (Per Second) for Azure:")
    print(lowest_azure_price if lowest_azure_price is not None else "No Azure data available.")

# Example usage
directory_path = './out'  # Update to your path
result_df = read_json_files_to_dataframe(directory_path)
# print(result_df.head())
find_lowest_spot_price(result_df)


Lowest Spot Price for AWS:
AvailabilityZone                       us-east-1a
InstanceType                             m5.large
ProductDescription                     Linux/UNIX
SpotPrice                                  0.0291
Timestamp               2025-03-20T01:32:34+00:00
source_file                              aws.json
currencyCode                                  NaN
tierMinimumUnits                              NaN
retailPrice                                   NaN
unitPrice                                     NaN
armRegionName                                 NaN
location                                      NaN
effectiveStartDate                            NaN
effectiveEndDate                              NaN
meterId                                       NaN
meterName                                     NaN
productId                                     NaN
skuId                                         NaN
productName                                   NaN
skuName                

In [11]:
import os
import pandas as pd
import json

def read_json_files_to_dataframe(directory):
    dataframes = []
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r') as file:
                    data = json.load(file)
                    
                    # Handle specific JSON structures
                    if 'Items' in data:
                        df = pd.DataFrame(data['Items'])
                        df['BillingCurrency'] = data.get('BillingCurrency', None)
                        df['CustomerEntityId'] = data.get('CustomerEntityId', None)
                        df['CustomerEntityType'] = data.get('CustomerEntityType', None)

                        # Convert Azure unit prices to per-second prices if unit is in hours
                        df['NormalizedUnitPrice'] = df.apply(
                            lambda x: x['unitPrice'] / 3600 if x.get('unitOfMeasure') == '1 Hour' else x['unitPrice'], axis=1
                        )
                    elif 'SpotPriceHistory' in data:
                        df = pd.DataFrame(data['SpotPriceHistory'])
                        df['SpotPrice'] = pd.to_numeric(df['SpotPrice'])
                    else:
                        df = pd.DataFrame([data])
                    
                    df['source_file'] = filename  # Add column to track source
                    dataframes.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    # Combine all DataFrames into one
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    else:
        print("No JSON files found or failed to read.")
        return pd.DataFrame()

def find_lowest_spot_price(df):
    # Extract AWS and Azure data
    aws_df = df[df['source_file'] == 'aws.json']
    azure_df = df[df['source_file'] == 'azure.json']

    # Find lowest SpotPrice for AWS
    lowest_aws_price = aws_df.loc[aws_df['SpotPrice'].idxmin()] if not aws_df.empty else None

    # Find lowest NormalizedUnitPrice for Azure
    lowest_azure_price = azure_df.loc[azure_df['NormalizedUnitPrice'].idxmin()] if not azure_df.empty else None

    print("Lowest Spot Price for AWS:")
    if lowest_aws_price is not None:
        print(f"SpotPrice: {lowest_aws_price['SpotPrice']}, AvailabilityZone: {lowest_aws_price['AvailabilityZone']}, ProductDescription: {lowest_aws_price['ProductDescription']}")
    else:
        print("No AWS data available.")

    print("\nLowest Spot Price (Per Second) for Azure:")
    if lowest_azure_price is not None:
        print(f"UnitPrice (Per Second): {lowest_azure_price['NormalizedUnitPrice']}, Location: {lowest_azure_price['location']}, ProductName: {lowest_azure_price['productName']}")
    else:
        print("No Azure data available.")

    # Compare and print the lowest among AWS and Azure
    if lowest_aws_price is not None and lowest_azure_price is not None:
        if lowest_aws_price['SpotPrice'] < lowest_azure_price['NormalizedUnitPrice']:
            print("\nOverall Lowest Price: AWS")
            print(f"SpotPrice: {lowest_aws_price['SpotPrice']}, AvailabilityZone: {lowest_aws_price['AvailabilityZone']}, ProductDescription: {lowest_aws_price['ProductDescription']}")
        else:
            print("\nOverall Lowest Price: Azure")
            print(f"UnitPrice (Per Second): {lowest_azure_price['NormalizedUnitPrice']}, Location: {lowest_azure_price['location']}, ProductName: {lowest_azure_price['productName']}")
    elif lowest_aws_price is not None:
        print("\nOverall Lowest Price: AWS")
    elif lowest_azure_price is not None:
        print("\nOverall Lowest Price: Azure")
    else:
        print("\nNo pricing data available.")

# Example usage
directory_path = './out'  # Update to your path
result_df = read_json_files_to_dataframe(directory_path)
# print(result_df.head())
find_lowest_spot_price(result_df)

Lowest Spot Price for AWS:
SpotPrice: 0.0291, AvailabilityZone: us-east-1a, ProductDescription: Linux/UNIX

Lowest Spot Price (Per Second) for Azure:
UnitPrice (Per Second): 0.00010569444444444444, Location: AU Central 2, ProductName: Virtual Machines Esv6 Series Windows

Overall Lowest Price: Azure
UnitPrice (Per Second): 0.00010569444444444444, Location: AU Central 2, ProductName: Virtual Machines Esv6 Series Windows
